# Patents linked to publications
_(Created in June 2019)_

This notebook is dependent on the following libraries

In [47]:
import dimcli
from dimcli.shortcuts import dslquery, dslqueryall
import pandas as pd
from pandas.io.json import json_normalize
#import plotly_express as px
#from plotly.offline import init_notebook_mode # needed for exports 
#init_notebook_mode(connected=True)
import time

___
# 1. Data Extraction and Preparation

In [48]:
GRIDID = "grid.13097.3c"


In [49]:
df_all_pubs_per_year = dslquery(f"""search publications where research_orgs.id="{GRIDID}" return year limit 1000""").as_dataframe()



In [51]:

# Get full list of reelvant publications linked to this organization
pubs_details = dslqueryall(f"""search publications where research_orgs.id="{GRIDID}" and year in [2015:2019] return publications[basics -author_affiliations +category_rcdc+category_for+researchers]""")
df_pubs_details = pubs_details.as_dataframe()



1000 / 37590
2000 / 37590
3000 / 37590
4000 / 37590
5000 / 37590
6000 / 37590
7000 / 37590
8000 / 37590
9000 / 37590
10000 / 37590
11000 / 37590
12000 / 37590
13000 / 37590
14000 / 37590
15000 / 37590
16000 / 37590
17000 / 37590
18000 / 37590
19000 / 37590
20000 / 37590
21000 / 37590
22000 / 37590
23000 / 37590
24000 / 37590
25000 / 37590
26000 / 37590
27000 / 37590
28000 / 37590
29000 / 37590
30000 / 37590
31000 / 37590
32000 / 37590
33000 / 37590
34000 / 37590
35000 / 37590
36000 / 37590
37000 / 37590
37590 / 37590


Create a new list of publications with simplifed FOR data

In [70]:
 #Create a new list of publications with simplifed FOR data
# Ensure that all pubs have a valid (empty, even) FOR value, also remove the FOR digit prefix to improve legibility
for x in pubs_details.publications:
    if not 'category_for' in x:
        x['category_for'] = ""
    else:
        x['category_for'] = [{'name' : x['name'][5:]} for x in x['category_for']] 
df_pubs_for = json_normalize(pubs_details.publications, record_path=['category_for'], meta=["id", "type", ["journal", "title"], "year"], errors='ignore', record_prefix='for_')





Get the patents infos using the publications 

In [71]:
# query structure is:
# d=dslquery(f"""search patents where publication_ids in ["pub.1111511314","pub.1113174788","pub.1111902055"] return patents limit 1000""")

from itertools import islice
def chunks_of(data, size):
    it = iter(data)
    chunk = list(islice(it, size))
    while chunk:
        yield chunk
        chunk = list(islice(it, size))

SIZE = 400

def run(ids_list):
    patents_out, n  = [], 0
    for chunk in chunks_of(ids_list, SIZE): # chunks of 200 args
        n += 1
        temp = ','.join(['"{}"'.format(i) for i in chunk])
        data = dslquery(f"""search patents where publication_ids in [{temp}] return patents[basics+publication_ids+id+FOR + RCDC +times_cited] limit 1000""")
        patents_out += data.patents
        print("[log] ", n*SIZE, " pubs > patents: ", len(data.patents))
        time.sleep(1)
    return patents_out
        
patents_list = run(list(df_pubs_details['id']))

[log]  400  pubs > patents:  0
[log]  800  pubs > patents:  0
[log]  1200  pubs > patents:  0
[log]  1600  pubs > patents:  0
[log]  2000  pubs > patents:  0
[log]  2400  pubs > patents:  0
[log]  2800  pubs > patents:  0
[log]  3200  pubs > patents:  0
[log]  3600  pubs > patents:  0
[log]  4000  pubs > patents:  0
[log]  4400  pubs > patents:  0
[log]  4800  pubs > patents:  0
[log]  5200  pubs > patents:  0
[log]  5600  pubs > patents:  0
[log]  6000  pubs > patents:  0
[log]  6400  pubs > patents:  0
[log]  6800  pubs > patents:  0
[log]  7200  pubs > patents:  0
[log]  7600  pubs > patents:  0
[log]  8000  pubs > patents:  0
[log]  8400  pubs > patents:  0
[log]  8800  pubs > patents:  0
[log]  9200  pubs > patents:  0
[log]  9600  pubs > patents:  0
[log]  10000  pubs > patents:  0
[log]  10400  pubs > patents:  0
[log]  10800  pubs > patents:  0
[log]  11200  pubs > patents:  0
[log]  11600  pubs > patents:  0
[log]  12000  pubs > patents:  0
[log]  12400  pubs > patents:  0
[lo

After going through all publications and extracting related patents, let's save the patents data so that we can use it later:

In [72]:
df_patent_details = pd.DataFrame().from_dict(patents_list)
# save to CSV
df_patent_details.to_csv("data/KCL1_2019_patents_by_id.csv")
# display top 3 rows
df_patent_details.head(3)

,FOR,RCDC,assignee_names,assignees,filing_status,granted_year,id,inventor_names,publication_date,publication_ids,times_cited,title,year
0,"[{'id': '2746', 'name': '0801 Artificial Intel...",NaN,"[Beckman Coulter Inc, BECKMAN COULTER INC]","[{'id': 'grid.418254.e', 'name': 'Beckman Coul...",Application,NaN,EP-2347352-A4,"[ZIGON, ROBERT, VANWINKLE, RACHEL, SMITH, JERE...",2018-01-24,"[pub.1019581843, pub.1092164786]",NaN,INTERACTIVE TREE PLOT FOR FLOW CYTOMETRY DATA,2009
1,"[{'id': '2464', 'name': '0305 Organic Chemistr...","[{'id': '501', 'name': 'Brain Disorders'}]",[Centre National de la Recherche Scientifique ...,"[{'id': 'grid.7429.8', 'acronym': 'INSERM', 'n...",Grant,2018.0,US-9944669-B2,"[Roberto Motterlini, Roberta FORESTI, Thierry ...",2018-04-17,"[pub.1030319901, pub.1034748881, pub.103717905...",NaN,"Fumarate-CO-releasing molecule hybrids, their ...",2015
2,NaN,NaN,"[Seagate Technology LLC, SEAGATE TECHNOLOGY LLC]","[{'id': 'grid.462839.2', 'name': 'Seagate (Uni...",Application,NaN,US-20160099016-A1,"[Yukiko Kubota, Timothy John Klemmer, Kai Chie...",2016-04-07,[pub.1091248898],NaN,MAGNETIC STACK INCLUDING MgO-Ti(ON) INTERLAYER,2015


We also want to normalise the assignees data in order to analyse it further later on (ps: first we need to make the assigness data structure more regular)

In [85]:
for x in patents_list:
    if not 'assignees' in x:
        x['assignees'] = []
df_patents_assignees = json_normalize(patents_list, record_path=['assignees'], meta=['id', 'publication_ids', 'year', 'title'], meta_prefix="grant_")
# save to CSV 
df_patents_assignees.to_csv("data/KCL2_2019_patents_by_assignees1.csv")
df_patents_assignees.head()

,acronym,country_name,id,name,grant_id,grant_publication_ids,grant_year,grant_title
0,NaN,United States,grid.418254.e,Beckman Coulter (United States),EP-2347352-A4,"[pub.1019581843, pub.1092164786]",2009,INTERACTIVE TREE PLOT FOR FLOW CYTOMETRY DATA
1,INSERM,France,grid.7429.8,French Institute of Health and Medical Research,US-9944669-B2,"[pub.1030319901, pub.1034748881, pub.103717905...",2015,"Fumarate-CO-releasing molecule hybrids, their ..."
2,UPEC,France,grid.410511.0,Paris 12 Val de Marne University,US-9944669-B2,"[pub.1030319901, pub.1034748881, pub.103717905...",2015,"Fumarate-CO-releasing molecule hybrids, their ..."
3,CNRS,France,grid.4444.0,French National Centre for Scientific Research,US-9944669-B2,"[pub.1030319901, pub.1034748881, pub.103717905...",2015,"Fumarate-CO-releasing molecule hybrids, their ..."
4,NaN,United States,grid.462839.2,Seagate (United States),US-20160099016-A1,[pub.1091248898],2015,MAGNETIC STACK INCLUDING MgO-Ti(ON) INTERLAYER


Let's do the same type of normalization for FOR codes 

In [86]:
# Normalize FOR codes as above
for x in patents_list:
    if not 'FOR' in x:
        x['FOR'] = ""
    else:
        x['FOR'] = [{'name' : x['name'][5:]} for x in x['FOR']] 
df_patents_for = json_normalize(patents_list, record_path=['FOR'], meta=['id', 'year', 'title'], errors='ignore', record_prefix='for_')
# save to CSV 
df_patents_for.to_csv("data/KCL2_patents_by_FOR.csv")
df_patents_for.head()

TypeError: 'set' object is not subscriptable

Finally, let's create another publications index, including only the KSU publications cited by the patents extracted above. 

In [65]:
# extract pubs from patents list
pubs_referenced_from_patents = []
for x in patents_list:
    if x['publication_ids']:
        pubs_referenced_from_patents += x['publication_ids']
# remove duplicates
pubs_referenced_from_patents = list(set(pubs_referenced_from_patents))
len(pubs_referenced_from_patents)

5012

Intersect list of publications from KSU with list of publications mentioned in patents

In [66]:
df_linked_pubs =  df_pubs_details[df_pubs_details['id'].isin(pubs_referenced_from_patents)]
df_linked_pubs.reset_index(drop=True)
# save to CSV 
df_linked_pubs.to_csv("data/KCL1_pubs_linked_to_patents.csv")
df_linked_pubs.head()


,category_for,category_rcdc,id,issue,journal,pages,researchers,title,type,volume,year
18590,"[{'id': '3120', 'name': '1109 Neurosciences'},...","[{'id': '542', 'name': 'Neurodegenerative'}, {...",pub.1090348791,Trends Pharmacol. Sci. 34 2013,"{'id': 'jour.1048218', 'title': 'Redox Biology'}",444-451,"[{'id': 'ur.0617061073.05', 'last_name': 'Rojo...",NRF2 deficiency replicates transcriptomic chan...,article,13,2017
18793,"[{'id': '2211', 'name': '11 Medical and Health...",NaN,pub.1092164786,10,"{'id': 'jour.1054998', 'title': 'European Jour...",1584-1797,"[{'id': 'ur.01220110375.49', 'last_name': 'Cos...",Guidelines for the use of flow cytometry and c...,article,47,2017
19001,"[{'id': '2921', 'name': '0912 Materials Engine...",NaN,pub.1091248898,35,"{'id': 'jour.1041450', 'title': 'ACS Applied M...",29857-29862,"[{'id': 'ur.01027245116.62', 'last_name': 'Bra...",Titanium Oxynitride Thin Films with Tunable Do...,article,9,2017
19382,"[{'id': '2211', 'name': '11 Medical and Health...","[{'id': '445', 'name': 'Heart Disease'}, {'id'...",pub.1084749935,NaN,"{'id': 'jour.1102215', 'title': 'Journal of Ph...",11-23,"[{'id': 'ur.012242621575.33', 'last_name': 'Hu...",Cardiac voltage-gated ion channels in safety p...,article,87,2017
20841,"[{'id': '2211', 'name': '11 Medical and Health...","[{'id': '439', 'name': 'Diagnostic Radiology'}...",pub.1070928630,6,"{'id': 'jour.1012368', 'title': 'Journal of Nu...",891-898,"[{'id': 'ur.01045554612.14', 'last_name': 'Gro...",Intraoperative Assessment of Tumor Resection M...,article,58,2017


Also, create a version of `df_linked_pubs` with simplified FOR codes so that it's easier to visualise.

In [67]:
df_linked_pubs_for =  df_pubs_for[df_pubs_for['id'].isin(pubs_referenced_from_patents)]
df_linked_pubs_for.reset_index(drop=True)
df_linked_pubs_for.head()


,for_name,id,type,category_for,journal.title,year
44023,sciences,pub.1090348791,article,"[{'name': 'sciences'}, {'name': ' and Health S...",Redox Biology,2017
44024,and Health Sciences,pub.1090348791,article,"[{'name': 'sciences'}, {'name': ' and Health S...",Redox Biology,2017
44553,and Health Sciences,pub.1092164786,article,"[{'name': ' and Health Sciences'}, {'name': 'o...",European Journal of Immunology,2017
44554,ology,pub.1092164786,article,"[{'name': ' and Health Sciences'}, {'name': 'o...",European Journal of Immunology,2017
45058,ials Engineering,pub.1091248898,article,"[{'name': 'ials Engineering'}, {'name': 'ring'}]",ACS Applied Materials & Interfaces,2017


In [61]:
for x in pubs_referenced_from_patents:
     if not 'research_orgs' in x:
        x['research_orgs'] = []
df_linked_orgs = json_normalize(pubs_referenced_from_patents, record_path=['research_orgs'], meta=['researchers', 'year', 'title'], meta_prefix="org_")
# save to CSV 
df_linked_orgs.to_csv("data/agritech_2019_patents_by_orgs.csv")
df_linked_orgs.head()

TypeError: 'str' object does not support item assignment

---
#### That's it - now it's time to create some visualizations!
___

# 2. Data Analysis

## 2.1 Publications

In [ ]:
px.bar(df_all_pubs_per_year, x="id", y="count", title="Total Publications per year from KSU")

In [ ]:
px.scatter(df_pubs_for, x="year", y="for_name", color="type", hover_name="for_name", marginal_x="histogram", marginal_y="histogram", height=800, title="Research areas of pubs in last 10 years (marginal subplots = X/Y totals)")

In [ ]:
px.bar(df_linked_pubs.groupby('year',  as_index=False).count(), x="year", y="id", title="Publications mentioned in patents, by year of publication")

In [ ]:
px.scatter(df_linked_pubs_for, x="year", y="for_name", color="type", hover_name="for_name", marginal_x="histogram", marginal_y="histogram", height=800, title="Research areas of pubs mentioned in patents (marginal subplots = X/Y totals)")

## 2.2 Patents

In [ ]:
px.bar(df_patent_details.groupby('year',  as_index=False).count(),  x="year", y="id",  title="Patents per Year citing publications from KSU")

In [ ]:
px.scatter(df_patents_for, x="year", y="for_name", hover_name="for_name", marginal_x="histogram", marginal_y="histogram", height=800, title="Research areas of patents (marginal subplots = X/Y totals)")

In [ ]:
px.scatter(df_patent_details, x="year", y="times_cited", hover_name="title",  hover_data=['id'], facet_col="filing_status", title="Patents per Year VS Timed Cited VS Filing Status")

## 2.3 Assignees of patents

In [ ]:
px.bar(df_patents_assignees.groupby('name',  as_index=False).count().sort_values(by="grant_id", ascending=False),  x="name", y="grant_id", hover_name="name",  height=400,  title="Assignees by No of Patents")

In [ ]:
px.scatter(df_patents_assignees,  x="grant_year", y="name", color="country_name", hover_name="name",  hover_data=["id"],  height=800, title="Assignees By Country and Year")